In [1]:
# TF-IDF vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Stage 3 Vectorisation and Extracting Keywords

**Goal** Use sklearn's TfidfVectorizer — fit it on a list containing just your document, get the feature names and their scores, sort by score descending, return the top N words             

**TF-IDF** converts a collection of text documents into a numerical matrix.
Each row is a document; each column is a token; each value measures how important
that token is to that document relative to the whole corpus.

**Formula:**
```
TF-IDF(t, d) = TF(t, d) × IDF(t)

TF(t, d)  = count of term t in document d  / total terms in d  (how frequent?)
IDF(t)    = log( N / df(t) )               (how rare across all N documents?)
```

**Intuition:**
- A word that appears in every document (e.g. "the") has IDF ≈ 0 → near-zero TF-IDF
- A word that appears only in one document gets a high IDF → high TF-IDF

**Why TF-IDF matters for NLP:**  
It converts text to a sparse vector that ML classifiers (SVM, logistic regression)
can operate on directly. It is the foundation of search engines and document similarity.                                                                                                                                         

In [3]:
def extract_keywords(text:str, top_k:int=10) -> list[str]:
    """
    Extract keywords from text using TF-IDF vectorization.
    """
    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(stop_words='english') # Remove common English stop words.
    # Fit and transform the text to get TF-IDF matrix.
    tfidf_matrix = vectorizer.fit_transform([text]) # A sparse matrix: shape = (n_documents, n_features).

    print(f'TF-IDF matrix shape: {tfidf_matrix.shape}  →  ({len([text])} docs × {tfidf_matrix.shape[1]} features)')
    print(f'Vocabulary size    : {len(vectorizer.vocabulary_)} unique tokens')

    # Get feature names (terms) and their corresponding TF-IDF scores.
    feature_names = vectorizer.get_feature_names_out() # Array of feature names (terms).
    tfidf_scores = tfidf_matrix.toarray()[0] # Convert sparse matrix to dense array and get the first (and only) row.
    # Get indices of top_k keywords based on TF-IDF scores.
    top_k_indices = np.argsort(tfidf_scores)[::-1][:top_k] # Sort scores in descending order and get top_k indices.
    # Get the corresponding keywords for the top_k indices.
    keywords = [feature_names[i] for i in top_k_indices if tfidf_scores[i] > 0] # Only include keywords with non-zero TF-IDF scores.
    return keywords

In [5]:
# Example usage
clean_text = "This is an example document for keyword extraction using TF-IDF. The TF-IDF method helps identify important terms in the document."
keywords = extract_keywords(clean_text.lower(), top_k=10)
print("Extracted Keywords:", keywords)

TF-IDF matrix shape: (1, 12)  →  (1 docs × 12 features)
Vocabulary size    : 12 unique tokens
Extracted Keywords: ['tf', 'idf', 'document', 'using', 'terms', 'method', 'keyword', 'important', 'identify', 'helps']


## The Problem with One Document

The IDF part requires a **corpus** — multiple documents to compare against. When you only have one document:

- Every word appears in 1 out of 1 documents  
- IDF = log(1/1) = **0** for every word  
- With smoothing it's all equal — every word gets the same IDF  
- You're left with just **term frequency**, which is basically a word count  

Word count alone doesn't tell you what a document is *about*. "The" will always rank near the top.

---

## What to Use Instead (Single Document)

| Method | How it works | Best for |
|--------|-------------|----------|
| **YAKE** | Uses word position, frequency, and co-occurrence within one doc | General single-doc extraction |
| **TextRank** | Graph algorithm — words that co-occur with many other words rank higher | Summarisation-style keywords |
| **KeyBERT** | Finds phrases most similar to the overall document embedding | Semantic, meaningful phrases |

Since this project already uses `sentence-transformers` for FAISS, **KeyBERT** is the most natural fit — it reuses the same embedding model.

---

## When TF-IDF *Is* the Right Tool

TF-IDF makes sense in this project if you fit it across your **entire seed corpus** rather than one document at a time:

```python
from sklearn.feature_extraction.text import TfidfVectorizer

all_docs = ["text of doc 1...", "text of doc 2...", "text of doc 3..."]
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(all_docs)